# UK Retail Sales Analysis — Python Portfolio Project

**Goal:** audit a fictional UK retailer’s sales, build KPIs, visualise trends, and identify opportunities. Dataset is synthetic.

Run cells top-to-bottom. In Google Colab, upload `sales_data.csv` to the notebook session first using the Files sidebar, or run the optional upload cell below. Column names are **lowercase**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Libraries loaded")

## 1. Load sales data

If working in Colab, upload the dataset by opening the Files panel, or uncomment the Colab upload lines.

In [ ]:
# Colab users may uncomment:
# from google.colab import files
# files.upload()

csv_path = Path("sales_data.csv")
if not csv_path.exists():
    csv_path = Path("sales_portfolio_project/sales_data.csv")
df = pd.read_csv(csv_path)
display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## 2. Data quality and cleaning

Check missing values, duplicates, date types, and accounting consistency. Do not drop duplicate product transactions just because products repeat: use unique `order_id` to check order identity.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print(df.dtypes)
print("Missing values by column:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())
print("Invalid order dates:", df["order_date"].isna().sum())
print("Sales/Cost/Profit mismatches:", (~np.isclose(df["sales"] - df["cost"], df["profit"], atol=0.02)).sum())
assert df["order_date"].notna().all(), "Review invalid dates before continuing"
assert df["order_id"].notna().all(), "Review missing order IDs before continuing"

## 3. Calculate the executive KPIs

In [ ]:
revenue = df["sales"].sum()
profit = df["profit"].sum()
orders = df["order_id"].nunique()
margin = profit / revenue * 100 if revenue else float("nan")
print(f"Revenue: £{revenue:,.2f}")
print(f"Profit: £{profit:,.2f}")
print(f"Orders: {orders:,}")
print(f"Profit margin: {margin:.2f}%")
kpis = pd.DataFrame([{"revenue": revenue, "profit": profit, "orders": orders, "profit_margin_pct": margin}])
kpis.to_csv(OUTPUT / "kpis.csv", index=False)

## 4. Revenue by month

In [ ]:
monthly = (df.assign(month=df["order_date"].dt.to_period("M").dt.to_timestamp())
           .groupby("month", as_index=False)[["sales", "profit"]].sum()
           .sort_values("month"))
display(monthly)
fig, ax = plt.subplots()
ax.plot(monthly["month"], monthly["sales"], marker="o")
ax.set(title="Monthly Sales Revenue", xlabel="Month", ylabel="Revenue (£)")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT / "monthly_revenue.png", dpi=160)
plt.show()
monthly.to_csv(OUTPUT / "monthly_performance.csv", index=False)

## 5. Revenue and margin by category

In [ ]:
category = df.groupby("category", as_index=False).agg(revenue=("sales", "sum"), profit=("profit", "sum"), orders=("order_id", "nunique"))
category["profit_margin_pct"] = category["profit"] / category["revenue"] * 100
category = category.sort_values("revenue", ascending=False)
display(category)
fig, ax = plt.subplots()
sns.barplot(data=category, x="revenue", y="category", ax=ax, color="steelblue")
ax.set(title="Revenue by Category", xlabel="Revenue (£)", ylabel="Category")
fig.tight_layout()
fig.savefig(OUTPUT / "category_revenue.png", dpi=160)
plt.show()
category.to_csv(OUTPUT / "category_performance.csv", index=False)

## 6. Regional sales and profitability

In [ ]:
regional = df.groupby("region", as_index=False).agg(revenue=("sales", "sum"), profit=("profit", "sum"))
regional["profit_margin_pct"] = regional["profit"] / regional["revenue"] * 100
regional = regional.sort_values("revenue", ascending=False)
display(regional)
ax = regional.set_index("region")[["revenue", "profit"]].plot.bar(rot=0)
ax.set(title="Regional Revenue and Profit", ylabel="£", xlabel="Region")
plt.tight_layout()
plt.savefig(OUTPUT / "regional_revenue_profit.png", dpi=160)
plt.show()
regional.to_csv(OUTPUT / "regional_performance.csv", index=False)

## 7. Top-selling products

In [ ]:
products = df.groupby("product", as_index=False).agg(revenue=("sales", "sum"), profit=("profit", "sum"), units=("quantity", "sum"))
top5 = products.sort_values("revenue", ascending=False).head(5)
display(top5)
fig, ax = plt.subplots()
sns.barplot(data=top5, x="revenue", y="product", ax=ax, color="steelblue")
ax.set(title="Top 5 Products by Revenue", xlabel="Revenue (£)", ylabel="Product")
fig.tight_layout()
fig.savefig(OUTPUT / "top_products.png", dpi=160)
plt.show()
products.sort_values("revenue", ascending=False).to_csv(OUTPUT / "product_performance.csv", index=False)

## 8. Year-over-year growth

In [ ]:
yearly = df.assign(year=df["order_date"].dt.year).groupby("year", as_index=False)[["sales", "profit"]].sum()
yearly["revenue_growth_pct"] = yearly["sales"].pct_change() * 100
display(yearly)
yearly.to_csv(OUTPUT / "yearly_performance.csv", index=False)

## 9. Portfolio interpretation

In your README, describe the main revenue driver, the best-performing region, the products at the top of the ranking, and any gap between sales and profitability. Explain that these are **synthetic transactions**, so business recommendations are practice conclusions rather than real company advice.

**Portfolio deliverables:** notebook, sales CSV, `outputs/` charts and summary CSVs, 3–5 written insights. Cross-check overall revenue and profit against your Excel and SQL projects.